In [ ]:
!pip install gradio sentence-transformers faiss-cpu transformers PyPDF2 torch
!pip install pypdf
import gradio as gr
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from pypdf import PdfReader


# --------------------------
# 1. Load and split PDF
# --------------------------
pdf_path = "/content/Eigenvalue Decomp..pdf"  # change this
reader = PdfReader(pdf_path)
text = " ".join(page.extract_text() for page in reader.pages if page.extract_text())


# Split text into chunks
chunk_size = 800
chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]


# --------------------------
# 2. Embed and build FAISS index
# --------------------------
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = embedder.encode(chunks)
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings, dtype=np.float32))


# --------------------------
# 3. Define model for answering
# --------------------------
llm = pipeline(
   "text-generation",
   model="mistralai/Mistral-7B-Instruct-v0.2",
   device_map="auto",
   max_new_tokens=128,
   temperature=0.2,
)


# --------------------------
# 4. Define helper functions
# --------------------------
def retrieve_context(query, k=3):
   query_vec = embedder.encode([query])
   distances, indices = index.search(np.array(query_vec, dtype=np.float32), k)
   return "\n\n".join(chunks[i] for i in indices[0])


def answer_question(question):
   context = retrieve_context(question)
   prompt = (
       f"You are a math assistant. Use the following reference text to answer the question.\n\n"
       f"Reference:\n{context}\n\n"
       f"Question: {question}\n\n"
       f"Answer clearly in Markdown with LaTeX syntax for math (using $$ ... $$ when appropriate)."
   )


   response = llm(prompt)[0]["generated_text"]
   # Trim the prompt repetition if needed
   answer = response[len(prompt):].strip()
   return answer


# --------------------------
# 5. Build Gradio UI
# --------------------------
with gr.Blocks() as demo:
   gr.Markdown("### 🧮 Math Q&A with PDF Knowledge (No LangChain, LaTeX Ready)")


   question = gr.Textbox(label="Ask a question:")
   output = gr.Markdown(label="Answer")


   submit = gr.Button("Get Answer")
   submit.click(fn=answer_question, inputs=question, outputs=output)


demo.launch()



